# Normalization & Dimensionality Reduction

Normalize, select highly variable genes, and compute reduced representations.

This notebook:
1. Normalizes counts (library-size normalization + log1p)
2. Selects highly variable genes
3. Scales data and runs PCA
4. Computes neighborhood graph, UMAP, and t-SNE
5. Saves processed h5ad

In [ ]:
# Parameters (injected by Papermill)
input_h5ad_path: str = "/data/results/experiment/01_qc_filtered.h5ad"
experiment_name: str = "experiment"
n_highly_variable: int = 2000
n_pcs: int = 50
n_neighbors: int = 15
bioaf_api_url: str = "http://localhost:8000"
experiment_id: str = ""

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False, figsize=(6, 4))

results_dir = f"/data/results/{experiment_name}"
figures_dir = os.path.join(results_dir, "figures", "normalization")
os.makedirs(figures_dir, exist_ok=True)

## 1. Load Filtered Data

In [ ]:
adata = sc.read_h5ad(input_h5ad_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")

## 2. Normalization

Library-size normalization to 10,000 counts per cell, followed by log1p transform.
Raw counts are preserved in `adata.raw`.

In [ ]:
# Store raw counts before normalization
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Freeze normalized data as .raw for later DE
adata.raw = adata
print("Normalization complete (normalize_total + log1p)")

## 3. Highly Variable Gene Selection

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=n_highly_variable, flavor="seurat_v3",
                            layer="counts")
print(f"HVGs selected: {adata.var['highly_variable'].sum()}")

fig, ax = plt.subplots(figsize=(8, 4))
sc.pl.highly_variable_genes(adata, show=False, ax=ax)
fig.savefig(os.path.join(figures_dir, "hvg_selection.png"), bbox_inches="tight")
plt.show()

## 4. Scale & PCA

In [ ]:
# Subset to HVGs for PCA
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=n_pcs, svd_solver="arpack")

# Copy PCA results back to full object
adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"]
adata.uns["pca"] = adata_hvg.uns["pca"]
adata.varm["PCs"] = np.zeros((adata.n_vars, n_pcs))
hvg_idx = np.where(adata.var["highly_variable"])[0]
adata.varm["PCs"][hvg_idx] = adata_hvg.varm["PCs"]

print(f"PCA computed with {n_pcs} components")

In [ ]:
# Variance explained by PCs
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sc.pl.pca_variance_ratio(adata, n_pcs=n_pcs, ax=axes[0], show=False)
axes[0].set_title("Variance Ratio")
sc.pl.pca(adata, color="total_counts", ax=axes[1], show=False)
axes[1].set_title("PCA (colored by UMI counts)")
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, "pca_overview.png"), bbox_inches="tight")
plt.show()

## 5. Neighborhood Graph, UMAP & t-SNE

In [ ]:
sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)
sc.tl.umap(adata)
sc.tl.tsne(adata, n_pcs=n_pcs)

print(f"Computed neighbors (k={n_neighbors}), UMAP, and t-SNE")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata, color="total_counts", ax=axes[0], show=False, title="UMAP")
sc.pl.tsne(adata, color="total_counts", ax=axes[1], show=False, title="t-SNE")
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, "umap_tsne.png"), bbox_inches="tight")
plt.show()

## 6. Save Processed Data

In [ ]:
output_path = os.path.join(results_dir, "02_normalized_dimreduced.h5ad")
adata.write_h5ad(output_path)
print(f"Saved to: {output_path}")
print(f"Shape: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Embeddings: {list(adata.obsm.keys())}")